In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_DIR / "training_data.csv", parse_dates=["date"])

print(f"Loaded: {len(df)} matches")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

Loaded: 49405 matches
Date range: 1872-11-30 to 2026-06-10


In [2]:
df = df[df["date"] >= "1990-01-01"].copy()

def get_outcome(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    elif row["home_score"] < row["away_score"]:
        return "away_win"
    else:
        return "draw"

df["outcome"]        = df.apply(get_outcome, axis=1)
df["is_friendly"]    = df["tournament"] == "Friendly"
df["is_competitive"] = ~df["is_friendly"]
df["is_worldcup"]    = df["tournament"] == "FIFA World Cup"
df["total_goals"]    = df["home_score"] + df["away_score"]

print(f"Matches from 1990 onwards: {len(df)}")
print(f"\nOutcome split:")
print(df["outcome"].value_counts())
print(f"\nFriendlies:  {df['is_friendly'].sum()}")
print(f"Competitive: {df['is_competitive'].sum()}")
print(f"World Cup:   {df['is_worldcup'].sum()}")

Matches from 1990 onwards: 32287

Outcome split:
outcome
home_win    15652
away_win     9045
draw         7590
Name: count, dtype: int64

Friendlies:  10829
Competitive: 21458
World Cup:   552


In [3]:
# all WC 2026 teams from our schedule
wc2026_groups = {
    "A": ["Mexico", "South Korea", "Czech Republic", "South Africa"],
    "B": ["Canada", "Bosnia and Herzegovina", "Switzerland", "Qatar"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

all_teams_in_data = pd.concat([df["home_team"], df["away_team"]])
team_counts = all_teams_in_data.value_counts()

print("WC 2026 team coverage in training data:")
print("-" * 55)
missing = []
low = []
for group, teams in wc2026_groups.items():
    for team in teams:
        count = team_counts.get(team, 0)
        if count == 0:
            status = "✗ MISSING"
            missing.append(team)
        elif count < 20:
            status = "⚠ LOW"
            low.append(team)
        else:
            status = "✓"
        print(f"  Group {group}  {team:<30} {count:>4} matches  {status}")

print()
if missing:
    print(f"MISSING - need name fix: {missing}")
if low:
    print(f"LOW DATA - under 20 matches: {low}")
if not missing and not low:
    print("All 48 teams have sufficient data ✓")

WC 2026 team coverage in training data:
-------------------------------------------------------
  Group A  Mexico                          656 matches  ✓
  Group A  South Korea                     554 matches  ✓
  Group A  Czech Republic                  360 matches  ✓
  Group A  South Africa                    454 matches  ✓
  Group B  Canada                          331 matches  ✓
  Group B  Bosnia and Herzegovina          283 matches  ✓
  Group B  Switzerland                     381 matches  ✓
  Group B  Qatar                           501 matches  ✓
  Group C  Brazil                          530 matches  ✓
  Group C  Morocco                         404 matches  ✓
  Group C  Haiti                           299 matches  ✓
  Group C  Scotland                        332 matches  ✓
  Group D  United States                   618 matches  ✓
  Group D  Paraguay                        396 matches  ✓
  Group D  Australia                       382 matches  ✓
  Group D  Turkey                 

In [4]:
wc = df[df["is_worldcup"]].copy()
wc["scoreline"] = (
    wc["home_score"].astype(int).astype(str) + "-" +
    wc["away_score"].astype(int).astype(str)
)

print("Most common WC scorelines:")
print(wc["scoreline"].value_counts().head(10))

print(f"\nAvg goals per match:")
print(f"  All matches:   {df['total_goals'].mean():.2f}")
print(f"  Friendlies:    {df[df['is_friendly']]['total_goals'].mean():.2f}")
print(f"  Competitive:   {df[df['is_competitive']]['total_goals'].mean():.2f}")
print(f"  World Cup:     {df[df['is_worldcup']]['total_goals'].mean():.2f}")

Most common WC scorelines:
scoreline
1-0    67
2-1    60
1-1    58
0-1    52
0-0    44
2-0    39
1-2    38
0-2    31
2-2    24
3-0    18
Name: count, dtype: int64

Avg goals per match:
  All matches:   2.76
  Friendlies:    2.55
  Competitive:   2.87
  World Cup:     2.52


In [5]:
wc = df[df["is_worldcup"]].copy()
wc["scoreline"] = (
    wc["home_score"].astype(int).astype(str) + "-" +
    wc["away_score"].astype(int).astype(str)
)

print("Most common WC scorelines (since 1990):")
print(wc["scoreline"].value_counts().head(10))

print(f"\nAvg goals per match:")
print(f"  All matches:   {df['total_goals'].mean():.2f}")
print(f"  Friendlies:    {df[df['is_friendly']]['total_goals'].mean():.2f}")
print(f"  Competitive:   {df[df['is_competitive']]['total_goals'].mean():.2f}")
print(f"  World Cup:     {df[df['is_worldcup']]['total_goals'].mean():.2f}")

Most common WC scorelines (since 1990):
scoreline
1-0    67
2-1    60
1-1    58
0-1    52
0-0    44
2-0    39
1-2    38
0-2    31
2-2    24
3-0    18
Name: count, dtype: int64

Avg goals per match:
  All matches:   2.76
  Friendlies:    2.55
  Competitive:   2.87
  World Cup:     2.52


In [6]:
out_path = PROCESSED_DIR / "matches_clean.parquet"
df.to_parquet(out_path, index=False)

print(f"Saved {len(df)} matches → {out_path}")
print("\nFiles ready:")
print("  matches_clean.parquet  ✓")
print("\nNext: 03_elo_ratings.ipynb")

Saved 32287 matches → ..\data\processed\matches_clean.parquet

Files ready:
  matches_clean.parquet  ✓

Next: 03_elo_ratings.ipynb
